# CI-GWAS vs REGENIE (Panel A) and age-pooled vs age-split MXP (Panel B)

This notebook produces a single **two-panel figure** (A and B, side by side).

- **Panel A:** CI-GWAS selected variants vs REGENIE GWAS, shown as −log10(p)
- **Panel B:** MXP correlations (age-pooled vs age-split) with highlights for CI-GWAS selection and GWAS significance

Paths below assume the same directory structure as in your scripts.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from scipy.stats import pearsonr

sns.set_context("paper")
plt.rcParams["figure.dpi"] = 140


## Paths + configuration

In [ ]:

CI_GWAS_TSV = Path("<CI_GWAS_FULL_RESULTS_TSV>")
CIGWAS_PATH = CI_GWAS_TSV

STEP2_ROOT = Path("<STEP2_ROOT>")

SBP_MXP_PATH  = Path("<SBP_MXP_PATH>")
SBP1_MXP_PATH = Path("<SBP1_MXP_PATH>")

MAIN_SETUPS = [
    "sbp_pre_post_1to60_no_cvd",
    "dbp_pre_post_1to60_no_cvd",
    "sbp_pre_post_1to60_age5_with_statins_no_cvd",
    "dbp_pre_post_1to60_age5_with_statins_no_cvd",
]

CIGWAS_FDR_THRESH = 0.05

PANEL_A_LABEL_RSIDS = ["rs77870048", "rs62053262"]

PANEL_A_LIM = (0.0, 20.0)

SETUP_POOLED = "sbp_pre_post_1to60_no_cvd"
PHENO_POOLED = "SBP_pre_ADJ"
SETUP_AGE5   = "sbp_pre_post_1to60_age5_with_statins_no_cvd"
PHENO_AGE1   = "SBP_pre_1_ADJ"

X_COL = PHENO_POOLED
Y_COL = PHENO_AGE1

GWAS_P_THRESH = 5e-8
GWAS_LOG10P_THRESH = float(-np.log10(GWAS_P_THRESH))

OUT_FIG_PDF = Path("figure_panels_A_B.pdf")


## Shared helpers (CI→REGENIE run mapping, file paths, caching)

In [ ]:
def build_ci_to_gwas_dict_pooled():
    return {
        "SBP_pre_ADJ": "pooled__SBP_pre__base",
        "DBP_pre_ADJ": "pooled__DBP_pre__base",
        "SBP_post_ADJ": "pooled__SBP_post__base+classes+pre",
        "DBP_post_ADJ": "pooled__DBP_post__base+classes+pre",
        "angiotensin_receptor_blocker": "pooled__CLASS_angiotensin_receptor_blocker__base+others+SBP_pre",
        "statin": "pooled__CLASS_statin__base+others+SBP_pre",
        "beta_blocker": "pooled__CLASS_beta_blocker__base+others+SBP_pre",
        "ACE_inhibitor": "pooled__CLASS_ACE_inhibitor__base+others+SBP_pre",
        "diuretic": "pooled__CLASS_diuretic__base+others+SBP_pre",
        "calcium_channel_blocker": "pooled__CLASS_calcium_channel_blocker__base+others+SBP_pre",
    }

def build_ci_to_gwas_dict_age5():
    d = {}
    for k in range(1, 6):
        d[f"SBP_pre_{k}_ADJ"]  = f"age5g{k}__SBP_pre__base"
        d[f"SBP_post_{k}_ADJ"] = f"age5g{k}__SBP_post__base+classes+pre"
        d[f"DBP_pre_{k}_ADJ"]  = f"age5g{k}__DBP_pre__base"
        d[f"DBP_post_{k}_ADJ"] = f"age5g{k}__DBP_post__base+classes+pre"

    drugs = [
        "angiotensin_receptor_blocker",
        "statin",
        "beta_blocker",
        "ACE_inhibitor",
        "diuretic",
        "calcium_channel_blocker",
    ]
    for k in range(1, 6):
        for drug in drugs:
            d[f"{drug}_{k}"] = f"age5g{k}__CLASS_{drug}__base+others+SBP_pre"
    return d

CI_TO_GWAS = {**build_ci_to_gwas_dict_pooled(), **build_ci_to_gwas_dict_age5()}

def step2_path(step2_root, run_id, chrom, trait):
    """REGENIE step2 file for a given (run_id, chrom, trait)."""
    trait = str(trait).removesuffix("_ADJ")
    return step2_root / run_id / f"chr{int(chrom)}" / f"{run_id}_chr{int(chrom)}_step2_{trait}.regenie"

def read_cigwas(path):
    df = pd.read_csv(path, sep="\t", dtype={"setup": "string", "phenotype": "string", "rsID": "string"})
    df["setup"] = df["setup"].astype(str)
    df["phenotype"] = df["phenotype"].astype(str)
    df["rsID"] = df["rsID"].astype(str)
    return df


## Fast lookup: REGENIE p-values for a list of (run_id, chr, trait, rsID)

In [ ]:
def lookup_regenie_pvalues(
    records,
    step2_root,
    chunksize=2_000_000,
):
    """
    Efficiently lookup REGENIE p-values for the given records.

    Expects columns:
      - regenie_run_id
      - chr
      - phenotype
      - rsID

    Returns a numpy array of p-values (float), same length/order as `records`.
    """
    out = np.full(len(records), np.nan, dtype="float64")

    # group by file identity
    gb = records.groupby(["regenie_run_id", "chr", "phenotype"], sort=False)
    for (run_id, chrom, phenotype), idx in gb.groups.items():
        run_id = str(run_id)
        chrom = int(chrom)
        phenotype = str(phenotype)

        rsids = records.loc[idx, "rsID"].astype(str)
        want = set(rsids.tolist())

        pth = step2_path(step2_root, run_id, chrom, phenotype)

        found_log10p = {}

        it = pd.read_csv(
            pth, sep=r"\s+",
            usecols=["ID", "LOG10P"],
            dtype={"ID": "string", "LOG10P": "float32"},
            chunksize=chunksize,
            engine="c",
        )

        for chunk in it:
            sub = chunk.loc[chunk["ID"].isin(want), ["ID", "LOG10P"]]
            if not sub.empty:
                for rid, lp in zip(sub["ID"].astype(str).tolist(), sub["LOG10P"].astype(float).tolist()):
                    found_log10p[rid] = lp
                want -= set(sub["ID"].astype(str).tolist())
                if not want:
                    break

        # fill outputs for this group
        log10ps = rsids.map(found_log10p).astype(float)
        out[idx] = np.power(10.0, -log10ps.to_numpy(dtype="float64"))

    return out


## Panel A: CI-GWAS selected variants vs REGENIE GWAS

In [ ]:
def plot_panel_a(
    ax,
    cigwas_path,
    step2_root,
    main_setups,
    fdr_thresh=0.05,
    label_rsids=None,
    lim=(0.0, 20.0),
):
    """Returns (pearson_r, pearson_p)."""
    ci = read_cigwas(cigwas_path)

    ci = ci.loc[ci["setup"].isin(list(main_setups))].copy()
    ci = ci.loc[pd.to_numeric(ci["p_fdr"], errors="coerce") < float(fdr_thresh)].copy()
    ci = ci.loc[ci["phenotype"].str.contains("_pre", case=False, regex=False)].copy()

    ci["regenie_run_id"] = ci["phenotype"].map(CI_TO_GWAS).astype("string")
    ci = ci.dropna(subset=["regenie_run_id", "chr", "p_value"]).copy()

    # lookup REGENIE p-values for these CI variants
    ci["regenie_p"] = lookup_regenie_pvalues(
        ci[["regenie_run_id", "chr", "phenotype", "rsID"]].copy(),
        step2_root=step2_root,
    )

    d = ci[["p_value", "regenie_p"]].copy()
    d["p_value"] = pd.to_numeric(d["p_value"], errors="coerce")
    d["regenie_p"] = pd.to_numeric(d["regenie_p"], errors="coerce")
    d = d.dropna(subset=["p_value", "regenie_p"]).copy()

    eps = 1e-300
    d = d[(d["p_value"] > 0) & (d["regenie_p"] > 0)].copy()
    d["ci_mlog10p"] = -np.log10(np.clip(d["p_value"].to_numpy(dtype=float), eps, 1.0))
    d["gwas_mlog10p"] = -np.log10(np.clip(d["regenie_p"].to_numpy(dtype=float), eps, 1.0))

    # scatter
    sns.scatterplot(
        data=d, x="ci_mlog10p", y="gwas_mlog10p",
        s=18, alpha=0.7, color="#9DB4A5", edgecolor=None, ax=ax
    )

    ax.plot([lim[0], lim[1]], [lim[0], lim[1]], linestyle="--", linewidth=1)
    ax.set_xlim(*lim)
    ax.set_ylim(*lim)
    ax.set_xlabel("CI-GWAS −log10(p)")
    ax.set_ylabel("REGENIE GWAS −log10(p)")
    ax.set_title("Selected variants: CI-GWAS vs standard GWAS (pre-treatment)")

    # pearson r
    r, p = pearsonr(d["ci_mlog10p"].to_numpy(), d["gwas_mlog10p"].to_numpy())
    ax.text(
        0.03, 0.97, f"Pearson r={r:.3f}\np={p:.2e}",
        transform=ax.transAxes, ha="left", va="top", fontsize=9
    )

    # optional labels
    if label_rsids:
        lab = ci.loc[ci["rsID"].isin(list(label_rsids)), ["rsID", "chr", "bp", "p_value", "regenie_p"]].copy()
        if not lab.empty:
            lab["ci_mlog10p"] = -np.log10(np.clip(pd.to_numeric(lab["p_value"], errors="coerce"), eps, 1.0))
            lab["gwas_mlog10p"] = -np.log10(np.clip(pd.to_numeric(lab["regenie_p"], errors="coerce"), eps, 1.0))
            lab["label"] = lab.apply(lambda r: f"{r['rsID']} chr{int(r['chr'])}:{int(r['bp'])}", axis=1)

            for rrow in lab.itertuples(index=False):
                ax.annotate(
                    rrow.label,
                    xy=(float(rrow.ci_mlog10p), float(rrow.gwas_mlog10p)),
                    xytext=(8, 0), textcoords="offset points",
                    ha="left", va="center", fontsize=9,
                )

    return float(r), float(p)


## Panel B: MXP correlation scatter with CI-GWAS and GWAS highlights

In [ ]:
# Styling (Panel B)
BASE_ALPHA = 0.06
BASE_SIZE  = 2
DOT_ALPHA  = 0.9
DOT_SIZE   = 16
TRI_ALPHA  = 0.6
TRI_SIZE   = 14
RING_ALPHA = 0.9
RING_SIZE  = 42

BASE_DOWNSAMPLE_FRAC = None

COL_BASE      = "#B8B8B8"
COL_CI_POOLED = "#1f77b4"
COL_CI_AGE1   = "#2ca02c"
COL_CI_BOTH   = "#7f0000"
COL_TRI_POOLED = "#f6b26b"
COL_TRI_AGE1   = "#93c47d"
COL_TRI_BOTH   = "#cc0000"
COL_RING      = "#000000"

def read_mxp_cols(path, cols):
    df = pd.read_csv(
        path, sep="\t", usecols=cols,
        dtype={"chr": "int32", "snp": "string"},
        engine="c", low_memory=False
    )
    df = df.rename(columns={"snp": "rsID"})
    df["rsID"] = df["rsID"].astype(str)
    return df

def collect_gwas_sig_variants_for_trait(
    step2_root,
    run_id,
    trait,
    log10p_thresh,
    cache_dir=Path("cache_step2_sig"),
    chunksize=2_000_000,
):
    """Return set of (chrom, rsID) with LOG10P >= log10p_thresh for the given (run_id, trait)."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    trait0 = str(trait).removesuffix("_ADJ")
    cache_path = cache_dir / f"sig__{run_id}__{trait0}__th{log10p_thresh:.3f}.pkl"
    if cache_path.exists():
        import pickle
        with cache_path.open("rb") as f:
            return pickle.load(f)

    sig = set()
    for chrom in range(1, 23):
        pth = step2_path(step2_root, run_id, chrom, trait)

        it = pd.read_csv(
            pth, sep=r"\s+",
            usecols=["ID", "LOG10P"],
            dtype={"ID": "string", "LOG10P": "float32"},
            chunksize=chunksize,
            engine="c",
        )
        for chunk in it:
            sub = chunk.loc[chunk["LOG10P"] >= float(log10p_thresh), ["ID"]]
            if not sub.empty:
                for rsid in sub["ID"].astype(str).tolist():
                    sig.add((chrom, rsid))

    import pickle
    with cache_path.open("wb") as f:
        pickle.dump(sig, f)

    return sig

def plot_panel_b(
    ax,
    cigwas_path,
    mxp_pooled_path,
    mxp_age1_path,
    step2_root,
    setup_pooled,
    pheno_pooled,
    setup_age5,
    pheno_age1,
    fdr_thresh,
    gwas_log10p_thresh,
    base_downsample_frac=None,
):
    # MXP data
    mxp_x = read_mxp_cols(mxp_pooled_path, cols=["chr", "snp", pheno_pooled])
    mxp_y = read_mxp_cols(mxp_age1_path,   cols=["chr", "snp", pheno_age1])

    mxp_x[pheno_pooled] = pd.to_numeric(mxp_x[pheno_pooled], errors="coerce")
    mxp_y[pheno_age1]   = pd.to_numeric(mxp_y[pheno_age1], errors="coerce")

    xy = mxp_x.merge(mxp_y, on=["chr", "rsID"], how="inner").dropna(subset=[pheno_pooled, pheno_age1]).copy()

    # CI-GWAS selection sets
    cig = read_cigwas(cigwas_path)
    cig["p_fdr"] = pd.to_numeric(cig["p_fdr"], errors="coerce")
    cig = cig.loc[cig["p_fdr"] < float(fdr_thresh)].copy()

    ci_pooled = cig.loc[(cig["setup"] == setup_pooled) & (cig["phenotype"] == pheno_pooled), ["rsID"]]
    ci_age1   = cig.loc[(cig["setup"] == setup_age5)   & (cig["phenotype"] == pheno_age1),   ["rsID"]]

    set_pooled = set(ci_pooled["rsID"].astype(str))
    set_age1   = set(ci_age1["rsID"].astype(str))

    both_ci = set_pooled & set_age1
    only_ci_pooled = set_pooled - both_ci
    only_ci_age1   = set_age1 - both_ci

    xy["ci_group"] = "none"
    xy.loc[xy["rsID"].isin(only_ci_pooled), "ci_group"] = "pooled_only"
    xy.loc[xy["rsID"].isin(only_ci_age1),   "ci_group"] = "age1_only"
    xy.loc[xy["rsID"].isin(both_ci),        "ci_group"] = "both"

    # GWAS-significant sets (trait-specific)
    run_pooled = CI_TO_GWAS[pheno_pooled]
    run_age1   = CI_TO_GWAS[pheno_age1]

    sig_pooled = collect_gwas_sig_variants_for_trait(
        step2_root=step2_root,
        run_id=run_pooled,
        trait=pheno_pooled,
        log10p_thresh=gwas_log10p_thresh,
    )
    sig_age1 = collect_gwas_sig_variants_for_trait(
        step2_root=step2_root,
        run_id=run_age1,
        trait=pheno_age1,
        log10p_thresh=gwas_log10p_thresh,
    )

    keys = list(zip(xy["chr"].astype(int).tolist(), xy["rsID"].astype(str).tolist()))
    xy["gwas_sig_pooled"] = [k in sig_pooled for k in keys]
    xy["gwas_sig_age1"]   = [k in sig_age1   for k in keys]

    xy["gwas_sig_group"] = "none"
    xy.loc[xy["gwas_sig_pooled"] & ~xy["gwas_sig_age1"], "gwas_sig_group"] = "pooled_only"
    xy.loc[~xy["gwas_sig_pooled"] &  xy["gwas_sig_age1"], "gwas_sig_group"] = "age1_only"
    xy.loc[xy["gwas_sig_pooled"] &  xy["gwas_sig_age1"],  "gwas_sig_group"] = "both"

    # Circle points that are selected by CI-GWAS AND GWAS-sig (within the matching run)
    xy["both_gwas_and_ci"] = (
        (xy["rsID"].isin(set_pooled) & xy["gwas_sig_pooled"]) |
        (xy["rsID"].isin(set_age1)   & xy["gwas_sig_age1"])
    )

    # Plot
    base = xy.loc[xy["ci_group"] == "none", [pheno_pooled, pheno_age1]]
    if base_downsample_frac is not None:
        base = base.sample(frac=float(base_downsample_frac), random_state=0)
    ax.scatter(base[pheno_pooled], base[pheno_age1], s=BASE_SIZE, alpha=BASE_ALPHA, color=COL_BASE, linewidths=0)

    sub = xy.loc[xy["ci_group"] == "pooled_only", [pheno_pooled, pheno_age1]]
    if not sub.empty:
        ax.scatter(sub[pheno_pooled], sub[pheno_age1], s=DOT_SIZE, alpha=DOT_ALPHA, color=COL_CI_POOLED, linewidths=0)

    sub = xy.loc[xy["ci_group"] == "age1_only", [pheno_pooled, pheno_age1]]
    if not sub.empty:
        ax.scatter(sub[pheno_pooled], sub[pheno_age1], s=DOT_SIZE, alpha=DOT_ALPHA, color=COL_CI_AGE1, linewidths=0)

    sub = xy.loc[xy["ci_group"] == "both", [pheno_pooled, pheno_age1]]
    if not sub.empty:
        ax.scatter(sub[pheno_pooled], sub[pheno_age1], s=DOT_SIZE, alpha=DOT_ALPHA, color=COL_CI_BOTH, linewidths=0)

    tri = xy.loc[xy["gwas_sig_group"] == "pooled_only", [pheno_pooled, pheno_age1]]
    if not tri.empty:
        ax.scatter(tri[pheno_pooled], tri[pheno_age1], s=TRI_SIZE, alpha=TRI_ALPHA, marker="^", color=COL_TRI_POOLED, linewidths=0)

    tri = xy.loc[xy["gwas_sig_group"] == "age1_only", [pheno_pooled, pheno_age1]]
    if not tri.empty:
        ax.scatter(tri[pheno_pooled], tri[pheno_age1], s=TRI_SIZE, alpha=TRI_ALPHA, marker="^", color=COL_TRI_AGE1, linewidths=0)

    tri = xy.loc[xy["gwas_sig_group"] == "both", [pheno_pooled, pheno_age1]]
    if not tri.empty:
        ax.scatter(tri[pheno_pooled], tri[pheno_age1], s=TRI_SIZE, alpha=TRI_ALPHA, marker="^", color=COL_TRI_BOTH, linewidths=0)

    ring = xy.loc[xy["both_gwas_and_ci"] == True, [pheno_pooled, pheno_age1]]
    if not ring.empty:
        ax.scatter(
            ring[pheno_pooled], ring[pheno_age1],
            s=RING_SIZE, alpha=RING_ALPHA, marker="o",
            facecolors="none", edgecolors=COL_RING, linewidths=0.8
        )

    mn = float(np.nanmin([xy[pheno_pooled].min(), xy[pheno_age1].min()]))
    mx = float(np.nanmax([xy[pheno_pooled].max(), xy[pheno_age1].max()]))
    ax.plot([mn, mx], [mn, mx], linestyle="--", linewidth=1)

    ax.set_xlabel("ρ(SNP, SBP) age-pooled")
    ax.set_ylabel("ρ(SNP, SBP) age-split")
    ax.set_title("MXP correlations: age-pooled vs age-split (SBP-pre)")

    legend = [
        Line2D([0],[0], marker="o", color="w", markerfacecolor=COL_BASE, markersize=6, label="all variants"),
        Line2D([0],[0], marker="o", color="w", markerfacecolor=COL_CI_POOLED, markersize=7, label="SBP-pre age-pooled graph"),
        Line2D([0],[0], marker="o", color="w", markerfacecolor=COL_CI_AGE1, markersize=7, label="SBP-pre (<50 years old) age-split graph"),
        Line2D([0],[0], marker="o", color="w", markerfacecolor=COL_CI_BOTH, markersize=7, label="both SBP-pre graph"),
        Line2D([0],[0], marker="^", color="w", markerfacecolor=COL_TRI_POOLED, markersize=8, label="SBP-pre age-pooled standard GWAS"),
        Line2D([0],[0], marker="^", color="w", markerfacecolor=COL_TRI_AGE1, markersize=8, label="SBP-pre (<50 years old) age-split standard GWAS"),
        Line2D([0],[0], marker="^", color="w", markerfacecolor=COL_TRI_BOTH, markersize=8, label="both SBP-pre standard GWAS"),
        Line2D([0],[0], marker="o", color=COL_RING, markerfacecolor="none", markersize=8, label="both graph and standard GWAS"),
    ]
    ax.legend(handles=legend, loc="upper left", frameon=True, fontsize=8)


## Make the final two-panel figure (A and B side-by-side)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Panel A
plot_panel_a(
    ax=axes[0],
    cigwas_path=CI_GWAS_TSV,
    step2_root=STEP2_ROOT,
    main_setups=MAIN_SETUPS,
    fdr_thresh=CIGWAS_FDR_THRESH,
    label_rsids=PANEL_A_LABEL_RSIDS,
    lim=PANEL_A_LIM,
)

# Panel B
plot_panel_b(
    ax=axes[1],
    cigwas_path=CIGWAS_PATH,
    mxp_pooled_path=SBP_MXP_PATH,
    mxp_age1_path=SBP1_MXP_PATH,
    step2_root=STEP2_ROOT,
    setup_pooled=SETUP_POOLED,
    pheno_pooled=PHENO_POOLED,
    setup_age5=SETUP_AGE5,
    pheno_age1=PHENO_AGE1,
    fdr_thresh=CIGWAS_FDR_THRESH,
    gwas_log10p_thresh=GWAS_LOG10P_THRESH,
    base_downsample_frac=BASE_DOWNSAMPLE_FRAC,
)

# Panel labels
for lab, ax in zip(["A", "B"], axes):
    ax.text(
        -0.08, 1.04, lab,
        transform=ax.transAxes,
        fontsize=16, fontweight="bold",
        va="top", ha="left"
    )

plt.tight_layout()
plt.savefig(OUT_FIG_PDF)
plt.show()

print(f"Saved: {OUT_FIG_PDF.resolve()}")
